In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# Load dataset
df = pd.read_csv("dataset.csv")

# Basic checks
print(df.info())
print(df.describe())

# Missing values
print("\nMissing values:\n", df.isnull().sum())

# Drop missing (sadə üsul)
df = df.dropna()

# Numeric & categorical columns
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(exclude=np.number).columns

# Constant columns drop
num_cols = [col for col in num_cols if df[col].nunique() > 1]

df_num = df[num_cols]

print("Numeric:", num_cols)
print("Categorical:", cat_cols)

### TASK 1

In [ ]:
# Correlation matrices
pearson_corr = df[num_cols].corr(method='pearson')
spearman_corr = df[num_cols].corr(method='spearman')

# Pairwise summary table
results = []

for i in range(len(num_cols)):
    for j in range(i+1, len(num_cols)):
        col1 = num_cols[i]
        col2 = num_cols[j]
        
        pearson_r, pearson_p = pearsonr(df[col1], df[col2])
        spearman_r, spearman_p = spearmanr(df[col1], df[col2])
        
        results.append({
            "Feature 1": col1,
            "Feature 2": col2,
            "Pearson r": pearson_r,
            "Pearson p": pearson_p,
            "Spearman ρ": spearman_r,
            "Spearman p": spearman_p
        })

summary_df = pd.DataFrame(results)
summary_df = summary_df.sort_values(by="Pearson r", key=abs, ascending=False)

summary_df.head(10)



## Task 1: Pearson and Spearman Correlation Analysis

### Strongest Linear Relationships

The strongest relationships in the dataset are observed between the following feature pairs:

- 2018 and 2020 (Pearson r ≈ 0.999, Spearman ρ ≈ 0.998)
- 2010 and 2020 (Pearson r ≈ 0.998, Spearman ρ ≈ 0.994)
- 2010 and 2018 (Pearson r ≈ 0.997, Spearman ρ ≈ 0.994)

These values are extremely close to 1, indicating a **very strong positive linear relationship** between these variables. This suggests that the features are highly related and likely represent the same underlying phenomenon measured across different years.

---

### Pearson vs Spearman Comparison

There is very little difference between the Pearson and Spearman correlation coefficients for all feature pairs:

- Pearson measures linear relationships
- Spearman measures monotonic (rank-based) relationships

Since both coefficients are very high and similar in value, this indicates that the relationships are both **linear and monotonic**.

---

### Interpretation of Relationships

The results show that:

- The relationships are strongly linear
- There is no significant evidence of non-linearity
- Outliers do not appear to significantly distort the relationships
- The ranking of values between variables is consistent

This means that as one variable increases, the others increase in a nearly perfect and predictable manner.

---

### Important Insight: Multicollinearity

The extremely high correlation values suggest the presence of **multicollinearity**, meaning that some features contain redundant information.

In practical applications, especially in machine learning models such as regression, highly correlated features can:

- Reduce model interpretability
- Cause instability in coefficient estimates

Therefore, one of the highly correlated variables could potentially be removed without significant loss of information.

---

### Data Quality Note

During the analysis, some features were found to have constant values (zero variance), which makes correlation undefined. These features were removed before computing Pearson and Spearman correlations to ensure valid and meaningful results.

### TASK 2

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

pivot_df = df.pivot_table(index=['REF_AREA', 'SEX'], 
                         columns='INDICATOR_LABEL', 
                         values='2020').dropna()

pivot_df.columns = [col[:15] + "..." if len(col) > 15 else col for col in pivot_df.columns]


pearson_corr = pivot_df.corr(method='pearson')
spearman_corr = pivot_df.corr(method='spearman')

mask = np.triu(np.ones_like(pearson_corr, dtype=bool))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))

# PEARSON HEATMAP
sns.heatmap(pearson_corr, 
            mask=mask, 
            annot=True, 
            fmt=".2f", 
            cmap='coolwarm', 
            center=0, 
            square=True, 
            linewidths=.5, 
            ax=ax1)
ax1.set_title('Pearson Correlation Matrix (Linear)', fontsize=16)

# SPEARMAN HEATMAP
sns.heatmap(spearman_corr, 
            mask=mask, 
            annot=True, 
            fmt=".2f", 
            cmap='coolwarm', 
            center=0, 
            square=True, 
            linewidths=.5, 
            ax=ax2)
ax2.set_title('Spearman Correlation Matrix (Monotonic)', fontsize=16)

plt.tight_layout()
plt.show()

## Task 2: Comparison of Pearson and Spearman Heatmaps

### Key Agreements

Both Pearson ($r$) and Spearman ($\rho$) correlation heatmaps show strong agreement across the main feature pairs.

- **Strong Linear Relationships:**  
  Both correlation measures are exceptionally high (greater than 0.90) for variables such as HCI, EYS, and LAYS.  
  This indicates a very strong positive relationship between education indicators and human capital, suggesting that as education levels increase, human capital also increases in a nearly linear manner.

- **Consistency in Direction:**  
  All correlations are positive in both heatmaps, showing a stable and consistent trend across the dataset.  
  This reinforces the reliability of the observed relationships.

---

### Key Differences and Insights

- **Non-Linear Relationships:**  
  Spearman’s $\rho$ values are consistently higher than Pearson’s $r$ for variables such as Probability of Survival (PSurv) and Test Scores.  

  **Interpretation:**  
  This suggests that the relationships are **monotonic but not strictly linear**.  
  For example, survival rates may reach a natural upper limit (plateau effect), resulting in a curved relationship that Spearman captures better than Pearson.

- **Outlier Sensitivity:**  
  Slightly lower Pearson values compared to Spearman indicate the possible presence of **outliers**.  
  These outliers may distort the linear correlation (Pearson), while Spearman remains more robust since it is based on rank rather than raw values.

---

### Conclusion

Overall, the heatmaps reveal that most relationships in the dataset are strong and positive.  
However, for certain variables—particularly health-related metrics—the relationships appear to follow a **curvilinear (non-linear) pattern**.

In such cases, Spearman correlation provides a more reliable measure of association, making it a valuable diagnostic tool when linear assumptions do not fully hold.

### TASK 3

In [ ]:
df = pd.read_csv("dataset.csv")
df = df.dropna(subset=['2020']) 

plot_df = df.pivot_table(index=['REF_AREA', 'SEX'], 
                         columns='INDICATOR_LABEL', 
                         values='2020').reset_index()


rename_dict = {
    'Human Capital Index (scale 0-1)': 'HCI',
    'Learning-Adjusted Years of School': 'LAYS',
    'Expected Years of School': 'EYS',
    'Harmonized Test Scores': 'Test_Score'
}
plot_df = plot_df.rename(columns=rename_dict)


top_pairs = [('LAYS', 'HCI'), ('LAYS', 'EYS'), ('Test_Score', 'HCI')]


for feat1, feat2 in top_pairs:
    if feat1 in plot_df.columns and feat2 in plot_df.columns:
        plt.figure(figsize=(10, 6))
        
        sns.scatterplot(data=plot_df, x=feat1, y=feat2, hue='SEX', alpha=0.6)
        
        sns.regplot(data=plot_df, x=feat1, y=feat2, scatter=False, color='black', label='Overall Trend')
        
        plt.title(f'Relationship: {feat1} vs {feat2}', fontsize=14)
        plt.legend(title='Gender', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

## Task 3: Scatterplot Diagnostic Analysis

### Regression Line Performance

- **Pattern Fit:**  
  The regression lines capture the overall patterns very effectively, particularly for the LAYS vs. HCI pair ($r \approx 0.99$), where the data points are tightly clustered around the fitted line.  

- **Linearity:**  
  This strong alignment indicates that the relationship between key education metrics and the Human Capital Index (HCI) is highly linear.  
  As a result, a linear regression model provides a reliable representation of the underlying data.

---

### Subgroup (Gender) Analysis

- **Slopes:**  
  The slopes for the Male (M) and Female (F) subgroups are nearly identical across all examined feature pairs.  
  This suggests that the effect of education on HCI (i.e., how much HCI increases with additional years of schooling or improved test scores) is consistent across genders.

- **Intercepts:**  
  There are no meaningful differences in intercepts between the subgroups.  
  This indicates that, for a given level of education or learning, both males and females achieve similar HCI values.

- **Simpson’s Paradox:**  
  No evidence of Simpson’s Paradox is observed.  
  The positive relationships seen in the aggregated data remain consistent within each gender subgroup, with no reversal in trend.

---

### Diagnostic Conclusion

The scatterplot analysis confirms that **Learning-Adjusted Years of Schooling (LAYS)** is the most stable and dominant predictor of the Human Capital Index.

Additionally, the similarity in slopes across gender subgroups suggests that the underlying relationship between education and human capital is uniform.  
This implies that the key drivers influencing HCI operate consistently across different demographic groups within this dataset.